In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import sys
sys.path.append('..')

from matplotlib.ticker import SymmetricalLogLocator, LogFormatterSciNotation
from matplotlib.ticker import AutoMinorLocator

In [ ]:
def generate_density(eigenvalues, weights, num_grid_points=int(1e5), sigma_squared=1e-5, boundary_margin=1e-3):

  def gaussian(x, x0, variance):
    return np.exp(-(x0 - x)**2 / (2.0 * variance)) / np.sqrt(2 * np.pi * variance)

  eigenvalues = np.array(eigenvalues).real
  weights = np.array(weights).real

  num_runs, num_eigvals = eigenvalues.shape

  lambda_max = np.mean(np.max(eigenvalues, axis=1), axis=0) + boundary_margin
  lambda_min = np.mean(np.min(eigenvalues, axis=1), axis=0) - boundary_margin

  grid_points = np.linspace(lambda_min, lambda_max, num=num_grid_points)
  grid_length = (lambda_max - lambda_min) / (num_grid_points - 1)

  sigma = sigma_squared * max(1, (lambda_max - lambda_min))

  # compute density
  densities = np.zeros((num_runs, num_grid_points), dtype=eigenvalues.dtype)
  for i in range(num_runs):
    for j in range(num_grid_points):
      x = grid_points[j]
      convolutions = gaussian(eigenvalues[i,:], x, sigma)
      densities[i,j] = np.sum(convolutions * weights[i,:])

  # average across runs
  densities = np.mean(densities, axis=0)
  densities = densities / (np.sum(densities) * grid_length)

  return densities, grid_points

In [ ]:
def get_title(system, Nf, pde_params):
    if system == 'convection':
        return rf'Convection, $n_{{\rm res}} = {Nf}, \beta = {pde_params}$'

In [ ]:
pdes = ['convection']
Nf_list = [100, 20000]
betas = [5, 50]
saved_results = {}
for pde in pdes:
    saved_results[pde] = {}
    for Nf in Nf_list:
        for beta in betas:
            hessian_before_file = os.path.join(
                './hessian',
                f"system_{pde}",
                f'N_f_{Nf}',
                f'beta_{beta}',
                "seed_1",
                'density_data_before.npz'
            )
            hessian_final_file = os.path.join(
                './hessian',
                f"system_{pde}",
                f'N_f_{Nf}',
                f'beta_{beta}',
                "seed_1",
                'density_data_final.npz'
            )
            saved_results[pde][f'({beta}, {Nf})'] = {}
            hessian_before = np.load(hessian_before_file, allow_pickle=True)
            hessian_final = np.load(hessian_final_file, allow_pickle=True)
            saved_results[pde][f'({beta}, {Nf})']["eigen_begin"] = hessian_before["eigen"]
            saved_results[pde][f'({beta}, {Nf})']["weight_begin"] = hessian_before["weight"]
            saved_results[pde][f'({beta}, {Nf})']["eigen_final"] = hessian_final["eigen"]
            saved_results[pde][f'({beta}, {Nf})']["weight_final"] = hessian_final["weight"]

In [ ]:
def plot_spectral_density(pdes, betas, Nf_list, saved_results, 
                          line_colors, line_styles, font_size=14, 
                          gaussian_variance=5e-5, y_cutoff=1e-10, 
                          folder_path='spectral_density_plots', filename='spectral_density',
                          show_plot=True, padding_ratio=0.1, fixed_linthresh=1):
    mpl.rcParams.update({'font.size': font_size})

    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

    legend_elements = [
        plt.Line2D([0], [0], color=line_colors['hessian_begin'], linestyle=line_styles['hessian_begin'], label='Before Training'),
        plt.Line2D([0], [0], color=line_colors['hessian_final'], linestyle=line_styles['hessian_final'], label='After Training')
    ]

    for pde in pdes:
        for beta in betas:
            for Nf in Nf_list:
                pde_result = saved_results[pde][f'({beta}, {Nf})']

                hessian_densities_before, hessian_grid_before = generate_density(
                    pde_result["eigen_begin"], 
                    pde_result["weight_begin"], 
                    sigma_squared=gaussian_variance
                )
                hessian_densities_final, hessian_grid_final = generate_density(
                    pde_result["eigen_final"], 
                    pde_result["weight_final"], 
                    sigma_squared=gaussian_variance
                )

                xlim_idx_hessian = np.argwhere(hessian_densities_before >= y_cutoff)[[0, -1]]
                xlim_idx_final = np.argwhere(hessian_densities_final >= y_cutoff)[[0, -1]]

                xlim_lower = min(hessian_grid_before[xlim_idx_hessian[0]], hessian_grid_final[xlim_idx_final[0]])
                xlim_upper = max(hessian_grid_before[xlim_idx_hessian[1]], hessian_grid_final[xlim_idx_final[1]])

                lower_exponent = np.floor(np.log10(np.abs(xlim_lower))) if xlim_lower != 0 else 0
                xlim_lower = -1 * (10 ** (lower_exponent + 1)) if xlim_lower < 0 else 10 ** lower_exponent
                if (xlim_lower <= 0) & (xlim_lower > -0.5):
                    xlim_lower = -0.5

                upper_exponent = np.floor(np.log10(np.abs(xlim_upper))) if xlim_upper != 0 else 0
                xlim_upper = np.ceil(xlim_upper / (10 ** upper_exponent)) * (10 ** upper_exponent)
                xlim_upper = xlim_upper * (1 + padding_ratio)

                fig, ax = plt.subplots(figsize=(6,4))
                ax.semilogy(hessian_grid_before, hessian_densities_before,
                            color=line_colors['hessian_begin'], linestyle=line_styles['hessian_begin'])
                ax.semilogy(hessian_grid_final, hessian_densities_final,
                            color=line_colors['hessian_final'], linestyle=line_styles['hessian_final'])

                ax.set_xscale('symlog', linthresh=fixed_linthresh)
                ax.xaxis.set_major_locator(SymmetricalLogLocator(base=10, linthresh=fixed_linthresh))
                ax.xaxis.set_major_formatter(LogFormatterSciNotation(base=10, linthresh=fixed_linthresh))

                ax.set_ylim(bottom=y_cutoff)
                ax.set_xlim(left=xlim_lower, right=xlim_upper)
                ax.set_ylabel('Density', fontname='Times New Roman')
                ax.set_xlabel('Eigenvalue', fontname='Times New Roman')
                # ax.set_title(get_title(pde, Nf, beta))

                fig.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, -0.05), ncol=2)
                fig.tight_layout(rect=[0, 0.03, 1, 0.95])

                full_filename = f"{filename}_{pde}_Nf_{Nf}_beta_{beta}.pdf"
                fig.savefig(os.path.join(folder_path, full_filename), bbox_inches='tight')

                if show_plot:
                    plt.show()
                else:
                    plt.close(fig)


In [ ]:
line_colors = {
  'hessian_begin': 'tab:blue',
  'hessian_final': 'tab:orange'
}
line_styles = {
  'hessian_begin': 'solid',
  'hessian_final': 'dashed'
}
plot_spectral_density(
    pdes, betas, Nf_list, saved_results, line_colors, line_styles, 
    font_size=10, gaussian_variance=5e-5, y_cutoff=1e-10,
    folder_path='spectral_density_plots', filename='spectral_density_convection'
)

In [ ]:
def plot_spectral_density_final(pdes, betas, Nf_list, saved_results, 
                                line_colors, line_styles, font_size=10, 
                                gaussian_variance=5e-5, y_cutoff=1e-10, 
                                folder_path='spectral_density_plots', filename='spectral_density_final',
                                show_plot=True, padding_ratio=0.1, fixed_linthresh=1):
    mpl.rcParams.update({'font.size': font_size})

    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

    legend_elements = [
        plt.Line2D([0], [0], color=line_colors['hessian_final'], linestyle=line_styles['hessian_final'], label='After Training')
    ]

    for pde in pdes:
        for beta in betas:
            for Nf in Nf_list:
                pde_result = saved_results[pde][f'({beta}, {Nf})']

                hessian_densities_final, hessian_grid_final = generate_density(
                    pde_result["eigen_final"], 
                    pde_result["weight_final"], 
                    sigma_squared=gaussian_variance
                )

                xlim_idx_final = np.argwhere(hessian_densities_final >= y_cutoff)[[0, -1]]
                xlim_lower = hessian_grid_final[xlim_idx_final[0]]
                xlim_upper = hessian_grid_final[xlim_idx_final[1]]

                lower_exponent = np.floor(np.log10(np.abs(xlim_lower))) if xlim_lower != 0 else 0
                xlim_lower = -1 * (10 ** (lower_exponent + 1)) if xlim_lower < 0 else 10 ** lower_exponent
                if (xlim_lower <= 0) & (xlim_lower > -0.5):
                    xlim_lower = -0.5

                upper_exponent = np.floor(np.log10(np.abs(xlim_upper))) if xlim_upper != 0 else 0
                xlim_upper = np.ceil(xlim_upper / (10 ** upper_exponent)) * (10 ** upper_exponent)
                xlim_upper = xlim_upper * (1 + padding_ratio)

                fig, ax = plt.subplots(figsize=(6,4))
                ax.semilogy(hessian_grid_final, hessian_densities_final,
                            color="blue", linestyle="solid")

                ax.set_xscale('symlog', linthresh=fixed_linthresh)
                ax.xaxis.set_major_locator(SymmetricalLogLocator(base=10, linthresh=fixed_linthresh))
                ax.xaxis.set_major_formatter(LogFormatterSciNotation(base=10, linthresh=fixed_linthresh))

                ax.set_ylim(bottom=y_cutoff)
                ax.set_xlim(left=xlim_lower, right=xlim_upper)
                ax.set_ylabel('Density', fontname='Times New Roman')
                ax.set_xlabel('Eigenvalue', fontname='Times New Roman')
                ax.set_title(get_title(pde, Nf, beta))

                fig.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, -0.05))
                fig.tight_layout(rect=[0, 0.03, 1, 0.95])

                full_filename = f"{filename}_{pde}_Nf_{Nf}_beta_{beta}.pdf"
                fig.savefig(os.path.join(folder_path, full_filename), bbox_inches='tight')

                if show_plot:
                    plt.show()
                else:
                    plt.close(fig)

In [ ]:
line_colors = {
  'hessian_begin': 'tab:blue',
  'hessian_final': 'tab:blue'
}
line_styles = {
  'hessian_begin': 'solid',
  'hessian_final': 'solid'
}
plot_spectral_density_final(
    pdes=pdes,
    betas=betas,
    Nf_list=Nf_list,
    saved_results=saved_results,
    line_colors=line_colors, 
    line_styles=line_styles,
    font_size=14,
    gaussian_variance=5e-5,
    y_cutoff=1e-10,
    folder_path='spectral_density_plots',
    filename='spectral_density_final',
    show_plot=True,   
    padding_ratio=0.1,
    fixed_linthresh=1
)